# asymptotics — Introduction

**asymptotics** is a semi-automated perturbation theory toolkit built on SymPy.

No symbol declarations needed — just write your equation as a string:

```python
from asymptotics import AlgebraicEquation

eq  = AlgebraicEquation("x**3 + eps*x - 1", dependent="x", small_param="eps")
sol = eq.expand_regular(order=3)
sol.show()
```

For coupled systems:

```python
from asymptotics import AlgebraicSystem

sys = AlgebraicSystem(
    equations   = ["x**2 + eps*y - 1", "y**2 + eps*x - 1"],
    dependents  = ["x", "y"],
    small_param = "eps",
)
sol = sys.expand_regular(order=3)
sol.show()
```

In [ ]:
from sympy import sqrt, series, simplify, latex, Eq
from IPython.display import display, Math
import numpy as np
from asymptotics import AlgebraicEquation, AlgebraicSystem

---
## Part 1 — Single equations

### 1.1 The simplest possible example

$$x - 1 + \varepsilon = 0$$

The exact solution is $x = 1 - \varepsilon$. Perturbation theory should recover this exactly.

In [ ]:
eq  = AlgebraicEquation("x - 1 + eps", dependent="x", small_param="eps")
sol = eq.expand_regular(order=2)
sol.show()

### 1.2 The classic cubic

$$x^3 + \varepsilon x - 1 = 0$$

In [ ]:
eq  = AlgebraicEquation("x**3 + eps*x - 1", dependent="x", small_param="eps")
sol = eq.expand_regular(order=3)
sol.show()

In [ ]:
# Numerical verification
def cubic_exact(eps_val):
    roots = np.roots([1, 0, eps_val, -1])
    return float(roots[np.isreal(roots)].real[0])

print(f"{'ε':>8}  {'exact':>14}  {'perturbation':>14}  {'error':>12}")
print("-" * 58)
for ev in [0.01, 0.05, 0.1, 0.2, 0.5, 1.0]:
    ex = cubic_exact(ev)
    ap = float(sol.composite.subs(sol.small_param, ev))
    print(f"{ev:>8.2f}  {ex:>14.8f}  {ap:>14.8f}  {abs(ex-ap):>12.2e}")

### 1.3 Inspecting intermediate steps

Every step is stored on the solution object.

In [ ]:
eq  = AlgebraicEquation("x**3 + eps*x - 1", dependent="x", small_param="eps")
sol = eq.expand_regular(order=3)

display(Math(r'\textbf{After ansatz substitution:}\quad' + latex(sol.substituted_equation) + r'= 0'))

display(Math(r'\textbf{Collected by power of }\varepsilon\textbf{:}'))
for k, coeff in sol.collected.items():
    lbl = r'\varepsilon^{' + str(k) + r'}' if k > 1 else (r'\varepsilon' if k == 1 else r'1')
    display(Math(r'\mathcal{O}(' + lbl + r'):\quad ' + latex(coeff) + r' = 0'))

### 1.4 Root selection

$$x^2 + \varepsilon x - 1 = 0$$

Two real branches — use `root_hint` to choose.

In [ ]:
eq_pos = AlgebraicEquation("x**2 + eps*x - 1", dependent="x", small_param="eps")
eq_neg = AlgebraicEquation("x**2 + eps*x - 1", dependent="x", small_param="eps", root_hint=-1)

sol_pos = eq_pos.expand_regular(order=3)
sol_neg = eq_neg.expand_regular(order=3)

print("Positive branch:")
sol_pos.show()
print("\nNegative branch:")
sol_neg.show()

### 1.5 Error handling

In [ ]:
from asymptotics import NoSmallParameterError, NoLeadingOrderSolutionError, OnlyComplexRootsError

# Missing small parameter
try:
    AlgebraicEquation("x**3 - 1", dependent="x", small_param="eps").expand_regular()
except NoSmallParameterError as e:
    print(e)

# O(1) not solvable (x = cos(x))
try:
    AlgebraicEquation("x - cos(x) - eps", dependent="x", small_param="eps").expand_regular()
except NoLeadingOrderSolutionError as e:
    print(e)

# Bad syntax
try:
    AlgebraicEquation("x^3 + eps*x - 1", dependent="x", small_param="eps")
except ValueError as e:
    print(e)

---
## Part 2 — Coupled systems

### 2.1 Symmetric algebraic system

$$x^2 + \varepsilon y - 1 = 0, \qquad y^2 + \varepsilon x - 1 = 0$$

By symmetry, $x(\varepsilon) = y(\varepsilon)$.

In [ ]:
sys = AlgebraicSystem(
    equations   = ["x**2 + eps*y - 1", "y**2 + eps*x - 1"],
    dependents  = ["x", "y"],
    small_param = "eps",
)

sol = sys.expand_regular(order=3)
sol.show()

In [ ]:
# Access per-variable results
display(Math(r'x(\varepsilon) = ' + latex(sol["x"].composite)))
display(Math(r'y(\varepsilon) = ' + latex(sol["y"].composite)))

from sympy import simplify
print("x = y by symmetry:", simplify(sol["x"].composite - sol["y"].composite) == 0)

### 2.2 Asymmetric system

$$x + \varepsilon y^2 - 1 = 0, \qquad y + \varepsilon x - 2 = 0$$

Unperturbed: $x_0 = 1$, $y_0 = 2$.

In [ ]:
sys = AlgebraicSystem(
    equations   = ["x + eps*y**2 - 1", "y + eps*x - 2"],
    dependents  = ["x", "y"],
    small_param = "eps",
)

sol = sys.expand_regular(order=3)
sol.show()

### 2.3 Three-variable system

$$x + \varepsilon y - 1 = 0, \qquad y + \varepsilon z - 2 = 0, \qquad z + \varepsilon x - 3 = 0$$

In [ ]:
sys = AlgebraicSystem(
    equations   = ["x + eps*y - 1", "y + eps*z - 2", "z + eps*x - 3"],
    dependents  = ["x", "y", "z"],
    small_param = "eps",
)

sol = sys.expand_regular(order=3)
sol.show()

---
## API quick reference

```python
from asymptotics import AlgebraicEquation, AlgebraicSystem

# Single equation
eq  = AlgebraicEquation("f(x, eps)", dependent="x", small_param="eps")
eq  = AlgebraicEquation("...", dependent="x", small_param="eps", root_hint=-1)
sol = eq.expand_regular(order=3)
sol.show()                    # full LaTeX hierarchy
sol.show(orders=[0, 1])       # specific orders
sol.composite                 # SymPy Expr
sol.small_param               # the eps symbol
sol[k].equation               # Eq at order k
sol[k].solution               # value of x_k

# Coupled system
sys = AlgebraicSystem(
    equations   = ["f1", "f2"],
    dependents  = ["x", "y"],
    small_param = "eps",
)
sol = sys.expand_regular(order=3)
sol.show()                    # all variables
sol["x"].composite            # composite for x
sol["x"][k].solution          # x_k
sol["y"].show()               # hierarchy for y only
```